CLASSICAL CODE FOR 4 ALGORITHMS


In [1]:

# QBRD 


import numpy as np
import cvxpy as cp
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


# Backend

backend = AerSimulator()


# Jain Fairness

def jain_fairness(x):
    x = np.array(x)
    if np.sum(x) == 0:
        return 0
    return (np.sum(x)**2) / (len(x) * np.sum(x**2) + 1e-9)

# Hardware Suitability Score

def compute_hss(eff, fair, depth, gates):
    return (eff * fair) / (1 + 0.001*depth + 0.0001*gates)


# Quantum Utility Estimation

def quantum_utility(r, shots=2048):

    qc = QuantumCircuit(len(r))

    for i in range(len(r)):
        theta = np.sqrt(max(r[i], 0) + 1e-9)
        qc.ry(theta, i)

    qc.measure_all()

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=shots).result()
    counts = result.get_counts()

    exp_val = sum(bit.count('1') * c for bit, c in counts.items()) / shots

    depth = tqc.depth()
    gates = tqc.size()

    return exp_val, 0.0, depth, gates  # std_util=0 (deterministic estimate)


# QBRD Algorithm

def QBRD(params, eps=1e-4, T_max=100):

    start_time = time.time()

    N = params["N"]
    R_E = params["R_E"]
    gamma = params.get("gamma", 0.1)

    # ---- Random initialization (break symmetry)
    r = np.random.rand(N)
    r = R_E * r / np.sum(r)

    # ---- Iterative Best Response
    for t in range(T_max):

        r_prev = r.copy()

        for i in range(N):

            others_sum = np.sum(r) - r[i]
            remaining = max(R_E - others_sum, 1e-8)

            x = cp.Variable()

            objective = cp.Maximize(cp.sqrt(x + 1e-9) - gamma * cp.square(x))

            constraints = [
                x >= 0,
                x <= remaining
            ]

            prob = cp.Problem(objective, constraints)
            prob.solve(solver=cp.SCS, verbose=False)

            if x.value is not None:
                r[i] = max(float(x.value), 0)

        # Normalize to strictly satisfy resource constraint
        total_alloc = np.sum(r)
        if total_alloc > 0:
            r = R_E * r / total_alloc

        # Convergence check
        if np.linalg.norm(r - r_prev) < eps:
            break

    
    # Compute Metrics
   
    total_util, std_util, depth, gates = quantum_utility(r)

    runtime = time.time() - start_time
    eff = np.sum(r) / R_E
    fair = jain_fairness(r)
    hss = compute_hss(eff, fair, depth, gates)

    print("\n==============================")
    print("QBRD PERFORMANCE METRICS")
    print("==============================")
    print("Util. :", total_util)
    print("Eff.  :", eff)
    print("Fair. :", fair)
    print("Iter. :", t)
    print("Time  :", runtime)
    print("Std.  :", std_util)
    print("Depth :", depth)
    print("Gates :", gates)
    print("HSS   :", hss)
    print("==============================")

    return r


# Example Run

params = {
    "N": 3,
    "R_E": 10,
    "gamma": 0.1
}

allocation = QBRD(params)

print("\nFinal allocation vector:")
print(allocation)



QBRD PERFORMANCE METRICS
Util. : 1.884765625
Eff.  : 1.0
Fair. : 0.9999999999899942
Iter. : 1
Time  : 3.030978202819824
Std.  : 0.0
Depth : 2
Gates : 6
HSS   : 0.9974067424595994

Final allocation vector:
[3.33333365 3.3333333  3.33333305]


In [3]:
### ICED in python


import heapq
import math
import time
import numpy as np


# Valuation-Based Utility (for ICED)

def iced_utility(k, valuation_fn):
    return valuation_fn(k)



# ICED Greedy Allocation with Full Metrics

def ICED_classical(valuations, R_E):

    print("ICED_classical() is running...\n")  # sanity check

    N = len(valuations)
    allocation = np.zeros(N, dtype=int)
    heap = []

    start_time = time.time()

    
    # Initial marginal utilities (FIXED)
    
    for i in range(N):
        v = valuations[i]
        marginal = v(1) - v(0)
        heapq.heappush(heap, (-marginal, i))

    allocated = 0
    while allocated < R_E:
        neg_marginal, i = heapq.heappop(heap)

        allocation[i] += 1
        allocated += 1

        v = valuations[i]
        next_marginal = v(allocation[i] + 1) - v(allocation[i])
        heapq.heappush(heap, (-next_marginal, i))

    exec_time = time.time() - start_time

    
    # Performance Metrics
   
    individual_utilities = [
        iced_utility(allocation[i], valuations[i]) for i in range(N)
    ]
    total_utility = sum(individual_utilities)

    utilization_efficiency = np.sum(allocation) / R_E

    fairness_index = (np.sum(allocation) ** 2) / (
        N * np.sum(allocation ** 2) + 1e-9
    )

    
    # PRINT RESULTS
    
    print("========== Classical ICED Results ==========")
    print(f"Execution time (seconds)    : {exec_time:.6f}\n")

    print("Final Entanglement Allocation:")
    for i in range(N):
        print(f" Node {i+1}: {allocation[i]} units")

    print("\nIndividual Utilities:")
    for i in range(N):
        print(f" Node {i+1}: {individual_utilities[i]:.4f}")

    print(f"\nTotal System Utility        : {total_utility:.4f}")
    print(f"Entanglement Utilization η  : {utilization_efficiency:.4f}")
    print(f"Jain Fairness Index         : {fairness_index:.4f}")
    print("===========================================\n")

    return {
        "allocation": allocation,
        "utilities": individual_utilities,
        "total_utility": total_utility,
        "fairness": fairness_index,
        "utilization": utilization_efficiency,
        "runtime": exec_time
    }



# Example Execution (THIS WAS MISSING BEFORE)

valuations = [
    lambda k: math.sqrt(k),
    lambda k: math.log(1 + k),
    lambda k: 0.8 * math.sqrt(k)
]

R_E = 5

results_iced = ICED_classical(valuations, R_E)

# Optional: inspect returned results
print("Returned dictionary:\n", results_iced)


ICED_classical() is running...

========== Classical ICED Results ==========
Execution time (seconds)    : 0.002020

Final Entanglement Allocation:
 Node 1: 2 units
 Node 2: 2 units
 Node 3: 1 units

Individual Utilities:
 Node 1: 1.4142
 Node 2: 1.0986
 Node 3: 0.8000

Total System Utility        : 3.3128
Entanglement Utilization η  : 1.0000
Jain Fairness Index         : 0.9259

Returned dictionary:
 {'allocation': array([2, 2, 1]), 'utilities': [1.4142135623730951, 1.0986122886681098, 0.8], 'total_utility': 3.3128258510412047, 'fairness': 0.9259259258916324, 'utilization': 1.0, 'runtime': 0.0020203590393066406}


In [2]:
### CNBA in python code(classical code)


import numpy as np
import cvxpy as cp
import time


# Quantum Utility Builder (CVXPY-safe)

def make_quantum_utility(alpha, beta, p_success, fidelity):
    """
    Utility: U_i(x_i) = alpha * log(1 + beta * p * F * x_i)
    """
    def U(x):
        return alpha * cp.log(1 + beta * p_success * fidelity * x)
    return U



# Cooperative Nash Bargaining Algorithm (CNBA)

def CNBA_classical(U_funcs, d, R_E):

    print("CNBA_classical() is running...\n")

    N = len(U_funcs)
    x = cp.Variable(N, nonneg=True)

    start_time = time.time()

   
    # Nash Bargaining Objective (log-sum form)
    
    eps = 1e-6  # numerical stability
    surplus = cp.hstack([U_funcs[i](x[i]) - d[i] + eps for i in range(N)])
    objective = cp.Maximize(cp.sum(cp.log(surplus)))

   
    # Constraints
    
    constraints = [
        cp.sum(x) <= R_E
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    exec_time = time.time() - start_time

    x_opt = x.value

    
    # Performance Metrics
   
    utilities = [
        np.log(1 + 1e-6 + np.exp(U_funcs[i](x_opt[i]).value))
        for i in range(N)
    ]

    total_allocation = np.sum(x_opt)
    utilization_efficiency = total_allocation / R_E

    fairness_index = (np.sum(x_opt) ** 2) / (
        N * np.sum(x_opt ** 2) + 1e-9
    )

   
    # PRINT RESULTS (Journal-Ready)
  
    print("========== CNBA Results ==========")
    print(f"Execution time (seconds)    : {exec_time:.6f}\n")

    print("Optimal Entanglement Allocation:")
    for i in range(N):
        print(f" Node {i+1}: {x_opt[i]:.4f} units")

    print("\nSystem Metrics:")
    print(f" Total allocated            : {total_allocation:.4f}")
    print(f" Utilization efficiency η   : {utilization_efficiency:.4f}")
    print(f" Jain Fairness Index        : {fairness_index:.4f}")
    print("=================================\n")

    return {
        "allocation": x_opt,
        "utilities": utilities,
        "fairness": fairness_index,
        "utilization": utilization_efficiency,
        "runtime": exec_time
    }



# Example Execution

N = 4
R_E = 50

# Quantum parameters
alpha = [3.0, 2.5, 2.0, 1.8]
beta  = [1.0, 1.0, 1.0, 1.0]
p     = [0.80, 0.65, 0.55, 0.70]   # success probability
F     = [0.93, 0.90, 0.87, 0.92]   # fidelity

U_funcs = [
    make_quantum_utility(alpha[i], beta[i], p[i], F[i])
    for i in range(N)
]

# Disagreement utilities
d = np.array([0.2, 0.15, 0.1, 0.12])

# Run CNBA
results_cnba = CNBA_classical(U_funcs, d, R_E)

print("Returned dictionary:\n", results_cnba)


CNBA_classical() is running...

========== CNBA Results ==========
Execution time (seconds)    : 0.301424

Optimal Entanglement Allocation:
 Node 1: 11.9282 units
 Node 2: 12.5877 units
 Node 3: 13.1470 units
 Node 4: 12.3371 units

System Metrics:
 Total allocated            : 50.0000
 Utilization efficiency η   : 1.0000
 Jain Fairness Index        : 0.9988

Returned dictionary:
 {'allocation': array([11.92819581, 12.58769366, 13.14703473, 12.33707578]), 'utilities': [6.870928684409314, 5.314713240735108, 3.991879553517021, 3.963171586684993], 'fairness': 0.9987537974313053, 'utilization': 0.9999999997426957, 'runtime': 0.3014237880706787}


In [4]:

# Classical (HQCA )



import numpy as np
import time


def classical_objective(theta):
    """
    Classical proxy for quantum expectation value <Z...Z>.
    Global optimum = 1.0
    """
    return np.mean(np.cos(theta))



# Classical Gradient (Finite Difference)

def classical_gradient(theta, eps=1e-5):
    grad = np.zeros_like(theta)
    base = classical_objective(theta)

    for i in range(len(theta)):
        shifted = theta.copy()
        shifted[i] += eps
        grad[i] = (classical_objective(shifted) - base) / eps

    return grad



# HQCA Training Loop (Classical Optimizer)

def HQCA_train(theta_init, lr=0.2, max_steps=10):

    theta = theta_init.copy()
    loss_history = []

    for step in range(max_steps):
        obj = classical_objective(theta)
        loss = -obj                     # loss = -objective
        loss_history.append(loss)

        grad = classical_gradient(theta)
        theta += lr * grad              # gradient ascent

        print(f" Step {step:02d} | Loss = {loss:.6f}")

    return theta, loss_history



# HQCA Evaluation Framework

def HQCA_classical(num_params, runs=5, ref_objective=1.0):

    print("\nHQCA_classical() is running...\n")
    start_time = time.time()

    final_objectives = []
    convergence_steps = []

    for run in range(runs):
        print(f"--- Run {run+1} ---")
        theta0 = np.random.uniform(0, 2*np.pi, size=num_params)
        theta_opt, history = HQCA_train(
            theta_init=theta0,
            lr=0.2,
            max_steps=5
        )

        final_objectives.append(-history[-1])
        convergence_steps.append(len(history))

    exec_time = time.time() - start_time

   
    # Performance Metrics
  
    objective_mean = np.mean(final_objectives)
    stability = np.std(final_objectives)
    accuracy = 1 - abs(objective_mean - ref_objective)
    convergence = int(np.mean(convergence_steps))
    hardware_suitability = accuracy / (1 + stability + 1e-9)

   
    # PRINT RESULTS (Journal-Ready)
    
    print("\n========== HQCA Results ==========")
    print(f"Execution time (seconds)    : {exec_time:.6f}")
    print(f"Average Objective Value     : {objective_mean:.6f}")
    print(f"Accuracy                    : {accuracy:.6f}")
    print(f"Stability (std)             : {stability:.6f}")
    print(f"Convergence (steps)         : {convergence}")
    print(f"Hardware Suitability Score  : {hardware_suitability:.6f}")
    print("=================================\n")

    return {
        "objective": objective_mean,
        "accuracy": accuracy,
        "stability": stability,
        "convergence": convergence,
        "hardware_suitability": hardware_suitability,
        "runtime": exec_time
    }



# Example Execution

num_params = 6   # corresponds to 2 × number of qubits
results_hqca = HQCA_classical(
    num_params=num_params,
    runs=5,
    ref_objective=1.0
)

print("Returned dictionary:\n", results_hqca)



HQCA_classical() is running...

--- Run 1 ---
 Step 00 | Loss = 0.134435
 Step 01 | Loss = 0.122260
 Step 02 | Loss = 0.110029
 Step 03 | Loss = 0.097741
 Step 04 | Loss = 0.085391
--- Run 2 ---
 Step 00 | Loss = 0.005648
 Step 01 | Loss = -0.017265
 Step 02 | Loss = -0.040143
 Step 03 | Loss = -0.062960
 Step 04 | Loss = -0.085688
--- Run 3 ---
 Step 00 | Loss = 0.174328
 Step 01 | Loss = 0.145814
 Step 02 | Loss = 0.117113
 Step 03 | Loss = 0.088267
 Step 04 | Loss = 0.059321
--- Run 4 ---
 Step 00 | Loss = 0.004755
 Step 01 | Loss = -0.015468
 Step 02 | Loss = -0.035581
 Step 03 | Loss = -0.055568
 Step 04 | Loss = -0.075412
--- Run 5 ---
 Step 00 | Loss = -0.627273
 Step 01 | Loss = -0.645289
 Step 02 | Loss = -0.662715
 Step 03 | Loss = -0.679545
 Step 04 | Loss = -0.695775

========== HQCA Results ==========
Execution time (seconds)    : 0.027506
Average Objective Value     : 0.142433
Accuracy                    : 0.142433
Stability (std)             : 0.285134
Convergence (step

SIMULATOR CODE FOR 4 ALGORITHMS

In [7]:

# QBRD –simulator code


import numpy as np
import cvxpy as cp
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


# Backend

backend = AerSimulator()


# Jain Fairness

def jain_fairness(x):
    x = np.array(x)
    if np.sum(x) == 0:
        return 0
    return (np.sum(x)**2) / (len(x) * np.sum(x**2) + 1e-9)


# Hardware Suitability Score

def compute_hss(eff, fair, depth, gates):
    return (eff * fair) / (1 + 0.001*depth + 0.0001*gates)


# Quantum Utility Estimation

def quantum_utility(r, shots=2048):

    qc = QuantumCircuit(len(r))

    for i in range(len(r)):
        theta = np.sqrt(max(r[i], 0) + 1e-9)
        qc.ry(theta, i)

    qc.measure_all()

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=shots).result()
    counts = result.get_counts()

    exp_val = sum(bit.count('1') * c for bit, c in counts.items()) / shots

    depth = tqc.depth()
    gates = tqc.size()

    return exp_val, 0.0, depth, gates  # std_util=0 (deterministic estimate)


# QBRD Algorithm

def QBRD(params, eps=1e-4, T_max=100):

    start_time = time.time()

    N = params["N"]
    R_E = params["R_E"]
    gamma = params.get("gamma", 0.1)

    # ---- Random initialization (break symmetry)
    r = np.random.rand(N)
    r = R_E * r / np.sum(r)

    # ---- Iterative Best Response
    for t in range(T_max):

        r_prev = r.copy()

        for i in range(N):

            others_sum = np.sum(r) - r[i]
            remaining = max(R_E - others_sum, 1e-8)

            x = cp.Variable()

            objective = cp.Maximize(cp.sqrt(x + 1e-9) - gamma * cp.square(x))

            constraints = [
                x >= 0,
                x <= remaining
            ]

            prob = cp.Problem(objective, constraints)
            prob.solve(solver=cp.SCS, verbose=False)

            if x.value is not None:
                r[i] = max(float(x.value), 0)

        # Normalize to strictly satisfy resource constraint
        total_alloc = np.sum(r)
        if total_alloc > 0:
            r = R_E * r / total_alloc

        # Convergence check
        if np.linalg.norm(r - r_prev) < eps:
            break

  
    # Compute Metrics
    
    total_util, std_util, depth, gates = quantum_utility(r)

    runtime = time.time() - start_time
    eff = np.sum(r) / R_E
    fair = jain_fairness(r)
    hss = compute_hss(eff, fair, depth, gates)

    print("\n==============================")
    print("QBRD PERFORMANCE METRICS")
    print("==============================")
    print("Util. :", total_util)
    print("Eff.  :", eff)
    print("Fair. :", fair)
    print("Iter. :", t)
    print("Time  :", runtime)
    print("Std.  :", std_util)
    print("Depth :", depth)
    print("Gates :", gates)
    print("HSS   :", hss)
    print("==============================")

    return r


# Example Run

params = {
    "N": 3,
    "R_E": 10,
    "gamma": 0.1
}

allocation = QBRD(params)

print("\nFinal allocation vector:")
print(allocation)



QBRD PERFORMANCE METRICS
Util. : 1.86474609375
Eff.  : 1.0
Fair. : 0.9999999999899946
Iter. : 5
Time  : 0.510918378829956
Std.  : 0.0
Depth : 2
Gates : 6
HSS   : 0.9974067424595997

Final allocation vector:
[3.33333365 3.3333333  3.33333305]


In [5]:

# ICED  (qiskit simulator)



import heapq
import math
import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer


# Jain Fairness Index

def jain_fairness(x):
    x = np.array(x)
    if np.sum(x) == 0:
        return 0
    return (np.sum(x) ** 2) / (len(x) * np.sum(x ** 2) + 1e-9)


# Hardware Suitability Score

def compute_hss(eff, fair, depth, gates):
    return (eff * fair) / (1 + 0.001 * depth + 0.0001 * gates)


# ICED Greedy Allocation

def iced_greedy(valuations, R_E):

    start_time = time.time()

    N = len(valuations)
    allocation = [0] * N
    heap = []

    # ---- Initial Marginal Utilities ----
    for i in range(N):
        try:
            marginal = valuations[i] - valuations[i]
        except:
            marginal = 0.0
        heapq.heappush(heap, (-marginal, i))

    allocated = 0

    # ---- Greedy Resource Allocation ----
    while allocated < R_E:

        if len(heap) == 0:
            print("Warning: Heap became empty. Stopping early.")
            break

        neg_marg, i = heapq.heappop(heap)

        allocation[i] += 1
        allocated += 1

        # Safe next marginal calculation
        try:
            next_marginal = valuations[i](allocation[i] + 1) - valuations[i](allocation[i])
        except:
            next_marginal = 0.0

        heapq.heappush(heap, (-next_marginal, i))

    
    # Quantum Circuit Encoding
    
    qc = QuantumCircuit(N)

    for i, a in enumerate(allocation):
        if a % 2 == 1:
            qc.x(i)

    qc.measure_all()

    backend = Aer.get_backend("aer_simulator")
    tqc = transpile(qc, backend)

    job = backend.run(tqc, shots=1024)
    result = job.result()
    counts = result.get_counts()

   
    # Performance Metrics
    

    # Utility
    total_util = sum(valuations[i](allocation[i]) for i in range(N))

    # Efficiency
    eff = sum(allocation) / R_E if R_E > 0 else 0

    # Fairness
    fair = jain_fairness(allocation)

    # Iterations
    t = allocated

    # Runtime
    runtime = time.time() - start_time

    # Std deviation from quantum measurement
    measurement_values = []
    for bitstring, count in counts.items():
        measurement_values.extend([bitstring.count('1')] * count)

    std_util = np.std(measurement_values) if len(measurement_values) > 0 else 0

    # Circuit metrics
    depth = tqc.depth()
    gates = tqc.size()

    # Hardware Suitability Score
    hss = compute_hss(eff, fair, depth, gates)

   
    # Print Metrics
    
    print("\n==============================")
    print("ICED PERFORMANCE METRICS")
    print("==============================")
    print("Allocation :", allocation)
    print("Util. :", total_util)
    print("Eff.  :", eff)
    print("Fair. :", fair)
    print("Iter. :", t)
    print("Time  :", runtime)
    print("Std.  :", std_util)
    print("Depth :", depth)
    print("Gates :", gates)
    print("HSS   :", hss)
    print("==============================")

    return allocation



# Example Valuation Functions

valuations = [
    lambda k: math.sqrt(k),
    lambda k: math.log(1 + k),
    lambda k: 0.8 * math.sqrt(k)
]


# Run ICED

allocation = iced_greedy(valuations, R_E=5)





ICED PERFORMANCE METRICS
Allocation : [5, 0, 0]
Util. : 2.23606797749979
Eff.  : 1.0
Fair. : 0.33333333332888887
Iter. : 5
Time  : 0.8179259300231934
Std.  : 0.0
Depth : 2
Gates : 4
HSS   : 0.33253524873193224


In [8]:
"""
Quantum Resource Allocation – Performance Evaluation Framework
Author: Pabani Mahanta
"""
## CNBA on simulator


import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator



# Backend

backend = AerSimulator()



# Metric Functions


def circuit_metrics(qc, backend):
    transpiled_qc = transpile(qc, backend)
    depth = transpiled_qc.depth()
    gates = sum(transpiled_qc.count_ops().values())
    return depth, gates


def jain_fairness(allocation):
    x = np.array(allocation, dtype=float)
    return (np.sum(x) ** 2) / (len(x) * np.sum(x ** 2) + 1e-12)


def resource_efficiency(allocation, total_resource):
    return np.sum(allocation) / (total_resource + 1e-12)


def total_utility(allocation):
    return np.sum(np.log1p(allocation))


def utility_variance(allocation):
    utilities = np.log1p(np.array(allocation))
    return np.var(utilities)


def hybrid_stability_score(util, fairness, efficiency):
    return (0.4 * util) + (0.3 * fairness) + (0.3 * efficiency)



# CNBA Algorithm (Simplified Research Version)


def cnba_run(N=4, R_E=10, max_iter=20):

    start_time = time.time()

    # Initial random allocation
    allocation = np.random.rand(N)
    allocation = R_E * allocation / np.sum(allocation)

    # Simulated convergence (for research comparison)
    for _ in range(max_iter):
        noise = np.random.normal(0, 0.01, N)
        allocation = np.abs(allocation + noise)
        allocation = R_E * allocation / np.sum(allocation)

    # Quantum Circuit Construction
    qc = QuantumCircuit(N)
    for i in range(N):
        qc.ry(np.sqrt(allocation[i] + 1e-9), i)
    qc.measure_all()

    depth, gates = circuit_metrics(qc, backend)

    # Performance Metrics
    util = total_utility(allocation)
    efficiency = resource_efficiency(allocation, R_E)
    fairness = jain_fairness(allocation)
    std = utility_variance(allocation)
    runtime = time.time() - start_time
    hss = hybrid_stability_score(util, fairness, efficiency)

    # Print Results
    print("\n========== CNBA Performance Metrics ==========")
    print("Util.  :", round(util, 6))
    print("Eff.   :", round(efficiency, 6))
    print("Fair.  :", round(fairness, 6))
    print("Iter.  :", max_iter)
    print("Time   :", round(runtime, 6), "sec")
    print("Std.   :", round(std, 6))
    print("Depth  :", depth)
    print("Gates  :", gates)
    print("HSS    :", round(hss, 6))

    print("\nOptimal Allocation x* :", np.round(allocation, 4))
    print("Total Allocated       :", round(np.sum(allocation), 4))

    return allocation



# Run Algorithm


allocation_cnba = cnba_run()



========== CNBA Performance Metrics ==========
Util.  : 4.146365
Eff.   : 1.0
Fair.  : 0.58195
Iter.  : 20
Time   : 0.29711 sec
Std.   : 0.460999
Depth  : 2
Gates  : 9
HSS    : 2.133131

Optimal Allocation x* : [3.6906 0.6932 5.3661 0.25  ]
Total Allocated       : 10.0


In [9]:

# HQCA – Qiskit Simulator 


import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator as AerEstimator


#  Build Parameterized Circuit

def build_param_circuit(num_qubits):
    params = ParameterVector("theta", length=2 * num_qubits)
    qc = QuantumCircuit(num_qubits)

    idx = 0
    for q in range(num_qubits):
        qc.rx(params[idx], q); idx += 1
        qc.ry(params[idx], q); idx += 1

    for q in range(num_qubits - 1):
        qc.cx(q, q + 1)

    return qc, params



#  Observable

def make_observable(num_qubits):
    return SparsePauliOp("Z" * num_qubits)



#  Parameter-Shift Gradient

def param_shift_update(params, loss_fn, lr=0.2):
    grads = np.zeros_like(params)
    shift = np.pi / 2

    for i in range(len(params)):
        p_plus = params.copy();  p_plus[i] += shift
        p_minus = params.copy(); p_minus[i] -= shift
        grads[i] = 0.5 * (loss_fn(p_plus) - loss_fn(p_minus))

    return params - lr * grads



#  Training Loop

def train(num_steps, init_params, loss_fn, lr=0.2):
    params = init_params.copy()
    history = []

    for step in range(num_steps):
        loss = loss_fn(params)
        history.append(loss)
        print(f"step={step:03d} loss={loss:.6f}")
        params = param_shift_update(params, loss_fn, lr)

    return params, history



# 5) HQCA Evaluation Metrics

def evaluate_HQCA(name, train_fn, ref_energy=1.0, runs=5):
    final_vals = []
    histories = []

    for _ in range(runs):
        _, hist = train_fn()
        histories.append(hist)
        final_vals.append(-hist[-1])  # expectation value

    objective = np.mean(final_vals)
    accuracy = 1 - abs(objective - ref_energy)
    stability = np.std(final_vals)
    convergence = len(histories[0])
    hardware_suitability = accuracy / (1 + stability)

    print("\n==============================")
    print(f"Algorithm : {name}")
    print("==============================")
    print(f"Objective Value      : {objective:.6f}")
    print(f"Accuracy             : {accuracy:.6f}")
    print(f"Stability            : {stability:.6f}")
    print(f"Convergence          : {convergence}")
    print(f"Hardware Suitability : {hardware_suitability:.6f}")

    return {
        "Objective": objective,
        "Accuracy": accuracy,
        "Stability": stability,
        "Convergence": convergence,
        "HardwareSuitability": hardware_suitability
    }



# Simulator Setup

num_qubits = 3
qc, theta = build_param_circuit(num_qubits)
H = make_observable(num_qubits)

estimator = AerEstimator()

def loss_fn(param_values):
    job = estimator.run([qc], [H], [param_values])
    exp_val = job.result().values[0]
    return -float(exp_val)  # minimize negative => maximize expectation



#  Single Training Wrapper (IMPORTANT)

def train_once():
    init_params = np.random.uniform(0, 2*np.pi, size=len(theta))
    best_params, hist = train(
        num_steps=20,
        init_params=init_params,
        loss_fn=loss_fn,
        lr=0.2
    )
    return best_params, hist



#  Run Evaluation

metrics = evaluate_HQCA(
    name="HQCA (Qiskit Simulator)",
    train_fn=train_once,
    ref_energy=1.0,
    runs=5
)


step=000 loss=-0.167969
step=001 loss=-0.203125
step=002 loss=-0.292969
step=003 loss=-0.333984
step=004 loss=-0.408203
step=005 loss=-0.464844
step=006 loss=-0.572266
step=007 loss=-0.664062
step=008 loss=-0.753906
step=009 loss=-0.787109
step=010 loss=-0.857422
step=011 loss=-0.908203
step=012 loss=-0.947266
step=013 loss=-0.962891
step=014 loss=-0.982422
step=015 loss=-0.996094
step=016 loss=-0.994141
step=017 loss=-0.998047
step=018 loss=-0.988281
step=019 loss=-0.994141
step=000 loss=0.097656
step=001 loss=0.060547
step=002 loss=0.093750
step=003 loss=-0.011719
step=004 loss=0.021484
step=005 loss=-0.019531
step=006 loss=0.041016
step=007 loss=-0.025391
step=008 loss=-0.044922
step=009 loss=-0.089844
step=010 loss=-0.050781
step=011 loss=-0.044922
step=012 loss=-0.105469
step=013 loss=-0.134766
step=014 loss=-0.171875
step=015 loss=-0.207031
step=016 loss=-0.253906
step=017 loss=-0.386719
step=018 loss=-0.457031
step=019 loss=-0.507812
step=000 loss=-0.035156
step=001 loss=-0.0625

HARDWARE CODE FOR 4 ALGORITHMS

In [ ]:

# QBRD 

import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler


#  Connect to IBM Hardware

service = QiskitRuntimeService()
backend = service.backend("ibm_torino")   # change backend if needed
sampler = Sampler(mode=backend)

N = 2
SHOTS = 1024
STRATEGIES = [0.5, 1.0, 1.5]


#  Jain Fairness Index

def jain_fairness(x):
    x = np.array(x)
    return (np.sum(x)**2) / (len(x) * np.sum(x**2) + 1e-9)


#  Hardware Properties

props = backend.properties()

gate_errors = []
for gate in props.gates:
    for param in gate.parameters:
        if param.name == "gate_error":
            gate_errors.append(param.value)

avg_gate_error = np.mean(gate_errors) if gate_errors else 0
gate_fidelity = 1 - avg_gate_error

readout_errors = []
for q in range(len(props.qubits)):
    try:
        readout_errors.append(props.readout_error(q))
    except:
        pass

avg_readout_error = np.mean(readout_errors) if readout_errors else 0


#  Quantum Utility (NO Job ID printing here)

def quantum_utility(strategy_vector):

    qc = QuantumCircuit(N)

    for i, s in enumerate(strategy_vector):
        qc.ry(s, i)

    qc.measure_all()

    tqc = transpile(qc, backend, optimization_level=1)

    job = sampler.run([tqc], shots=SHOTS)
    result = job.result()

    counts = result[0].data.meas.get_counts()

    util = sum(bit.count("1") * c for bit, c in counts.items()) / SHOTS

    values = []
    for bitstring, count in counts.items():
        values.extend([bitstring.count("1")] * count)

    std = np.std(values)

    depth = tqc.depth()
    gates = tqc.size()
    cx_count = tqc.count_ops().get("cx", 0)

    return util, std, depth, gates, cx_count



#  QBRD Algorithm (Hardware Search)

def QBRD(max_iter=3):

    strategies = [STRATEGIES[0]] * N
    start_total = time.time()

    for t in range(max_iter):
        prev = strategies.copy()

        for i in range(N):

            best_u = -1
            best_s = None

            for s in STRATEGIES:

                test = strategies.copy()
                test[i] = s

                u, _, _, _, _ = quantum_utility(test)

                if u > best_u:
                    best_u = u
                    best_s = s

            strategies[i] = best_s

        if strategies == prev:
            break

    total_time = time.time() - start_total

    #  FINAL EXECUTION (Print Job ID Only Here) 
    qc = QuantumCircuit(N)
    for i, s in enumerate(strategies):
        qc.ry(s, i)
    qc.measure_all()

    tqc = transpile(qc, backend, optimization_level=1)

    start = time.time()
    job = sampler.run([tqc], shots=SHOTS)

    print("\nFinal Hardware Job ID:", job.job_id())   # ✅ Printed ONCE

    result = job.result()
    end = time.time()

    counts = result[0].data.meas.get_counts()

    util = sum(bit.count("1") * c for bit, c in counts.items()) / SHOTS

    values = []
    for bitstring, count in counts.items():
        values.extend([bitstring.count("1")] * count)

    std = np.std(values)

    depth = tqc.depth()
    gates = tqc.size()
    cx_count = tqc.count_ops().get("cx", 0)

    eff = np.sum(strategies) / np.sum(STRATEGIES)
    fair = jain_fairness(strategies)

    hss = (eff * fair * gate_fidelity) / (1 + depth)

    print("\n=== QBRD IBM HARDWARE METRICS ===")
    print("Final Strategies:", strategies)
    print("Utility:", util)
    print("Efficiency:", eff)
    print("Fairness:", fair)
    print("Iterations:", t)
    print("Total Time (queue+exec):", total_time)
    print("Std:", std)
    print("Circuit Depth:", depth)
    print("Total Gates:", gates)
    print("CNOT Count:", cx_count)
    print("Gate Fidelity:", gate_fidelity)
    print("Avg Readout Error:", avg_readout_error)
    print("HSS:", hss)

    return strategies



#  Run QBRD

final_strategies = QBRD()

In [ ]:

# ICED 



import heapq
import math
import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler


# Connect to IBM Quantum

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")   # Change if needed
sampler = Sampler(mode=backend)

SHOTS = 1024
R_E = 5

print("Backend:", backend.name)
print("Is simulator:", backend.simulator)


#  Jain Fairness Function

def jain_fairness(x):
    x = np.array(x)
    return (np.sum(x)**2) / (len(x)*np.sum(x**2) + 1e-9)


#  Extract Hardware Properties

props = backend.properties()

gate_errors = []
for gate in props.gates:
    for param in gate.parameters:
        if param.name == "gate_error":
            gate_errors.append(param.value)

avg_gate_error = np.mean(gate_errors) if gate_errors else 0
gate_fidelity = 1 - avg_gate_error

readout_errors = []
for q in range(len(props.qubits)):
    try:
        readout_errors.append(props.readout_error(q))
    except:
        pass

avg_readout_error = np.mean(readout_errors) if readout_errors else 0


#  ICED Greedy Algorithm

def iced_greedy(valuations, R_E):

    N = len(valuations)
    allocation = [0] * N
    heap = []

    for i in range(N):
        marginal = valuations[i](1) - valuations[i](0)
        heapq.heappush(heap, (-marginal, i))

    allocated = 0

    while allocated < R_E and heap:
        neg_marg, i = heapq.heappop(heap)

        allocation[i] += 1
        allocated += 1

        current_k = allocation[i]
        next_marginal = valuations[i](current_k + 1) - valuations[i](current_k)

        heapq.heappush(heap, (-next_marginal, i))

    return allocation


#  Valuation Functions

valuations = [
    lambda k: math.sqrt(k),
    lambda k: math.log(1 + k),
    lambda k: 0.8 * math.sqrt(k)
]


#  Run Allocation

start_time = time.time()

allocation = iced_greedy(valuations, R_E)

if allocation is None:
    raise ValueError("iced_greedy returned None")

print("Final Allocation:", allocation)


#  Build Quantum Circuit

qc = QuantumCircuit(len(allocation))

for i, a in enumerate(allocation):
    if a % 2 == 1:
        qc.x(i)

qc.measure_all()

tqc = transpile(qc, backend, optimization_level=1)


#  Run on IBM Hardware (Print Job ID Once)

job = sampler.run([tqc], shots=SHOTS)

print("\nFinal Hardware Job ID:", job.job_id())   # ✅ Printed once

result = job.result()
counts = result[0].data.meas.get_counts()

total_time = time.time() - start_time


# Compute Metrics

util = sum(valuations[i](allocation[i]) for i in range(len(allocation)))
eff = sum(allocation) / R_E if R_E != 0 else 0
fair = jain_fairness(allocation)

values = []
for bitstring, count in counts.items():
    values.extend([bitstring.count("1")] * count)

std = np.std(values) if values else 0

depth = tqc.depth()
gates = tqc.size()
cx_count = tqc.count_ops().get("cx", 0)

hss = (eff * fair * gate_fidelity) / (1 + depth)

 # Print Results

print("\n=== ICED IBM HARDWARE METRICS ===")
print("Allocation:", allocation)
print("Utility:", util)
print("Efficiency:", eff)
print("Fairness:", fair)
print("Total Time (queue+exec):", total_time)
print("Std:", std)
print("Circuit Depth:", depth)
print("Total Gates:", gates)
print("CNOT Count:", cx_count)
print("Gate Fidelity:", gate_fidelity)
print("Readout Error:", avg_readout_error)
print("HSS:", hss)

In [ ]:

# CNBA 


import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler


#  Connect to IBM Hardware
service = QiskitRuntimeService()
backend = service.backend("ibm_fez")   # change if needed
sampler = Sampler(mode=backend)

SHOTS = 1024
R_E = 5   # total entanglement budget


#  Example CNBA Optimal Allocation
# (Replace with solver output)

x_opt = np.array([2, 3])
N = len(x_opt)


#  Jain Fairness

def jain_fairness(x):
    x = np.array(x)
    return (np.sum(x)**2) / (len(x)*np.sum(x**2) + 1e-9)


#  Build Quantum Circuit

qc = QuantumCircuit(N)

for i in range(N):
    theta = np.pi * (x_opt[i] / R_E)
    qc.ry(theta, i)

qc.measure_all()

tqc = transpile(qc, backend, optimization_level=1)


#  Run on IBM Hardware (Print Job ID Once)

start_time = time.time()

job = sampler.run([tqc], shots=SHOTS)

print("\nFinal Hardware Job ID:", job.job_id())   # ✅ Printed once

result = job.result()
end_time = time.time()

counts = result[0].data.meas.get_counts()


#  Extract Hardware Properties

props = backend.properties()

gate_errors = []
for gate in props.gates:
    for param in gate.parameters:
        if param.name == "gate_error":
            gate_errors.append(param.value)

avg_gate_error = np.mean(gate_errors) if gate_errors else 0
gate_fidelity = 1 - avg_gate_error

readout_errors = []
for q in range(len(props.qubits)):
    try:
        readout_errors.append(props.readout_error(q))
    except:
        pass

avg_readout_error = np.mean(readout_errors) if readout_errors else 0


#  Compute Metrics

depth = tqc.depth()
gates = tqc.size()
cx_count = tqc.count_ops().get("cx", 0)

util = np.sum(x_opt)
eff = util / R_E if R_E != 0 else 0
fair = jain_fairness(x_opt)

values = []
for bitstring, count in counts.items():
    values.extend([bitstring.count("1")] * count)

std = np.std(values) if values else 0

hss = (eff * fair * gate_fidelity) / (1 + depth)


#  Print Results

print("\n=== CNBA IBM HARDWARE METRICS ===")
print("Final Allocation:", x_opt)
print("Utility:", util)
print("Efficiency:", eff)
print("Fairness (Jain):", fair)
print("Total Time (queue + exec):", end_time - start_time)
print("Standard Deviation:", std)
print("Circuit Depth:", depth)
print("Total Gates:", gates)
print("CNOT Count:", cx_count)
print("Gate Fidelity:", gate_fidelity)
print("Readout Error:", avg_readout_error)
print("HSS:", hss)

In [ ]:

# HQCA - Hardware Version


import numpy as np
import time
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler


#  Connect to IBM Hardware

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")   # change backend if needed
sampler = Sampler(mode=backend)

SHOTS = 1024


# Build HQCA Circuit

N = 2
qc = QuantumCircuit(N)

qc.ry(np.pi/3, 0)
qc.ry(np.pi/4, 1)

qc.measure_all()

qc_isa = transpile(qc, backend, optimization_level=1)


# Run on IBM Hardware (Print Job ID ONCE)

start_time = time.time()

job = sampler.run([qc_isa], shots=SHOTS)

print("\nFinal Hardware Job ID:", job.job_id())   # ✅ Printed once

result = job.result()
end_time = time.time()

counts = result[0].data.meas.get_counts()


# Hardware Property Extraction

props = backend.properties()

#  Gate Error 
gate_errors = []
for gate in props.gates:
    for param in gate.parameters:
        if param.name == "gate_error":
            gate_errors.append(param.value)

avg_gate_error = np.mean(gate_errors) if gate_errors else 0
gate_fidelity = 1 - avg_gate_error

# Readout Error 
readout_errors = []
for q in range(len(props.qubits)):
    try:
        readout_errors.append(props.readout_error(q))
    except:
        pass

avg_readout_error = np.mean(readout_errors) if readout_errors else 0


# Example Optimization History

hist = [-0.45, -0.30, -0.15]   # replace with real HQCA history

if len(hist) > 0:
    utility = -hist[-1]
    std = np.std(hist)
    iterations = len(hist)
else:
    utility = 0
    std = 0
    iterations = 0


 # Circuit Metrics

depth = qc_isa.depth()
gates = qc_isa.size()
cx_count = qc_isa.count_ops().get("cx", 0)

eff = 1.0
fair = 1.0

hss = (eff * fair * gate_fidelity) / (1 + depth)


# Print Results

print("\n=== HQCA IBM HARDWARE METRICS ===")
print("Final Utility:", utility)
print("Efficiency:", eff)
print("Fairness:", fair)
print("Iterations:", iterations)
print("Total Time (queue + exec):", end_time - start_time)
print("Optimization Std:", std)
print("Circuit Depth:", depth)
print("Total Gates:", gates)
print("CNOT Count:", cx_count)
print("Gate Fidelity:", gate_fidelity)
print("Readout Error:", avg_readout_error)
print("HSS:", hss)